# Phase 1 — Spatial Setup: OSM Intersections + Road Network

**Goal:** Pull signalized intersection locations and road networks from OSM
for target counties, perform nearest-link joins, and classify by road hierarchy.

**Output:** GeoParquet files in `data/processed/`
- `intersections_{county}.parquet` — traffic signal points with nearest-link join
- `roads_{county}.parquet` — road network edges
- `boundary_{county}.parquet` — county boundary polygon

In [1]:
import sys
sys.path.insert(0, "..")

import os
import warnings
warnings.filterwarnings("ignore")

import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt

from src.data.load_osm import load_intersections, load_road_network
from src.utils.spatial_utils import nearest_link_join, load_county_boundary

In [2]:
# ── Config ─────────────────────────────────────────────────────────
COUNTY = "harris_tx"
CACHE_DIR = "../data/raw/osm"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Target: {COUNTY}")

Target: harris_tx


In [3]:
# ── Load county boundary ───────────────────────────────────────────
boundary = load_county_boundary(COUNTY, cache_dir=CACHE_DIR)
print(f"County boundary: {boundary.shape}")
print(boundary[['name', 'geometry']].head())

County boundary: (1, 17)
            name                                           geometry
0  Harris County  POLYGON ((-95.96085 30.16338, -95.95744 30.154...


In [4]:
# ── Load signalized intersections ─────────────────────────────────
intersections = load_intersections(COUNTY, cache_dir=CACHE_DIR)
print(f"\nIntersections: {len(intersections):,} traffic signals")
print(f"Columns: {list(intersections.columns)}")
intersections.head(3)


Intersections: 11,433 traffic signals
Columns: ['id', 'geometry', 'highway', 'traffic_signals', 'source', 'traffic_signals:direction', 'traffic_signals:arrow', 'direction', 'name', 'survey:date', 'note', 'placement', 'description', 'crossing', 'crossing:island', 'fixme', 'tactile_paving', 'button_operated', 'crossing:markings', 'crossing:signals', 'traffic_signals:countdown']


,id,geometry,highway,traffic_signals,source,traffic_signals:direction,traffic_signals:arrow,direction,name,survey:date,...,placement,description,crossing,crossing:island,fixme,tactile_paving,button_operated,crossing:markings,crossing:signals,traffic_signals:countdown
0,151367069,POINT (-95.78248 29.77214),traffic_signals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,151367496,POINT (-95.64535 29.88242),traffic_signals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,151367599,POINT (-95.18926 29.66511),traffic_signals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ── Load road network ──────────────────────────────────────────────
roads = load_road_network(COUNTY, cache_dir=CACHE_DIR)
print(f"\nRoad edges: {len(roads):,}")
print(f"Highway types: {roads['highway'].value_counts().head(10).to_dict()}")

roads_harris_tx: 253574 duplicate rows on ['osmid']



Road edges: 365,814
Highway types: {'residential': 248887, 'secondary': 44958, 'tertiary': 38052, 'secondary_link': 10300, 'primary': 8487, 'unclassified': 6777, 'primary_link': 2218, 'motorway_link': 2168, 'motorway': 1752, 'tertiary_link': 1654}


In [ ]:
# ── Nearest-link join ─────────────────────────────────────────────
joined = nearest_link_join(intersections, roads)
print(f"Joined: {len(joined):,} intersections with nearest road link")
print(f"\nDistance stats (meters):")
print(joined['distance_m'].describe())
joined.head(3)

In [ ]:
# ── Classify by road hierarchy ────────────────────────────────────
from src.utils.spatial_utils import classify_road_hierarchy

joined['road_class'] = joined['highway_link'].apply(classify_road_hierarchy)
print(f"\nRoad class distribution:")
print(joined['road_class'].value_counts())

In [ ]:
# ── Quick spatial plot ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1)
roads.plot(ax=ax, linewidth=0.3, color="gray", alpha=0.5, label="Roads")
intersections.plot(ax=ax, markersize=2, color="red", alpha=0.6, label="Traffic signals")
ax.set_title(f"{COUNTY} — Traffic Signals + Road Network", fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────
intersections.to_parquet(f"{OUTPUT_DIR}/intersections_{COUNTY}.parquet", index=False)
roads.to_parquet(f"{OUTPUT_DIR}/roads_{COUNTY}.parquet", index=False)
joined.to_parquet(f"{OUTPUT_DIR}/intersections_joined_{COUNTY}.parquet", index=False)
print(f"✅ Saved to {OUTPUT_DIR}/")

## Summary
- Total signalized intersections loaded from OSM
- Joined to nearest road link with distance
- Classified by road hierarchy
- Repeat for Travis County, then Mecklenburg and Hamilton